# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [33]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [34]:
# Load the dataset
df = pd.read_csv('data/AviationData.csv', encoding='latin-1', low_memory=False)

# Inspect the dataset
print("DataTypes & Missing Values:")
df.info()

print("\nSummary Statistics:")
df.describe(include='all')

print("Missing Values per Column:")
display(df.isna().sum())


DataTypes & Missing Values:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make      

Event.Id                      0
Investigation.Type            0
Accident.Number               0
Event.Date                    0
Location                     52
Country                     226
Latitude                  54507
Longitude                 54516
Airport.Code              38757
Airport.Name              36185
Injury.Severity            1000
Aircraft.damage            3194
Aircraft.Category         56602
Registration.Number        1382
Make                         63
Model                        92
Amateur.Built               102
Number.of.Engines          6084
Engine.Type                7096
FAR.Description           56866
Schedule                  76307
Purpose.of.flight          6192
Air.carrier               72241
Total.Fatal.Injuries      11401
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Uninjured            5912
Weather.Condition          4492
Broad.phase.of.flight     27165
Report.Status              6384
Publication.Date          13771
dtype: i

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [35]:
# Convert 'Event Date' to datetime and enforce a 40 year limit
df['Event.Date'] = pd.to_datetime(df['Event.Date'], errors='coerce')
df = df[df['Event.Date'].dt.year >= 1983].copy()

df = df[df['Amateur.Built'].astype(str).str.lower() == 'no'].copy()

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [36]:
# Metric for fatal accidents
InjuryColumns = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']
df[InjuryColumns] = df[InjuryColumns].fillna(0)

df['TotalPassengers'] = df[InjuryColumns].sum(axis=1)

# calculate the casualty rate
df['CasualtyRate'] = np.where(
    df['TotalPassengers'] > 0,
      (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']) / df['TotalPassengers'],
0
)


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [37]:
df['Aircraft.damage'] = df['Aircraft.damage'].astype(str).str.lower().str.strip()
df['IsDestroyed'] = (df['Aircraft.damage'] == 'destroyed').astype(int)

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [38]:
# Standardize 'Make' column
df['Make'] = df['Make'].astype(str).str.lower().str.strip()

# >= 50 occurrences
MakeCounts = df['Make'].value_counts()
FrequentMakes = MakeCounts[MakeCounts >= 50].index
df = df[df['Make'].isin(FrequentMakes)].copy()

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [39]:
# Drop row with missing aircraft model
df = df.dropna(subset=['Model']).copy()

# Standardize 'Model' column
df['Model'] = df['Model'].astype(str).str.lower().str.strip()

#combine 'Make' and 'Model' 
df['MakeModel'] = df['Make'] + ' ' + df['Model']
df['MakeModel'] = df['MakeModel'].str.lower().str.strip()

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [40]:
# Standardize strings and replace missing values
ColumnsToClean = ['Engine.Type', 'Weather.Condition', 'Purpose.of.flight', 'Broad.phase.of.flight']

for col in ColumnsToClean:
    df[col] = df[col].astype(str).str.lower().str.strip()
    df[col] = df[col].replace(['nan', 'unknown', 'unk'], np.nan)

    df['Number.of.Engines'] = pd.to_numeric(df['Number.of.Engines'], errors='coerce')

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [41]:
# Calculate % of missing values per column
MissingPercent = 0.60 
MissingCols = df.isnull().mean()

# Drop columns with more than 60% missing values
ColsToDrop = MissingCols[MissingCols > MissingPercent].index
CleanedDF = df.drop(columns=ColsToDrop)

print(f"Dropped: {len(ColsToDrop)} columns: {list(ColsToDrop)}")
print(f"Remaining columns: {CleanedDF.shape}")

Dropped: 6 columns: ['Latitude', 'Longitude', 'Aircraft.Category', 'FAR.Description', 'Schedule', 'Air.carrier']
Remaining columns: (70452, 29)


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [42]:
OutputFile = 'data/AviationData_Cleaned.csv'
CleanedDF.to_csv(OutputFile, index=False)
print(f"Cleaned dataset saved to: {OutputFile}")

Cleaned dataset saved to: data/AviationData_Cleaned.csv
